In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools.eval_measures import rmse, aic
import warnings

In [2]:
panel_data = pd.read_csv('panel_data_2025.csv', parse_dates=['date'])

In [3]:
panel_data.head()

,date,appid,name,player_count,twitch_count,review_sentiment,reddit_sentiment,post_count
0,2025-01-01,570,Dota 2,638080.0,24898.0,0.93,0.74,129
1,2025-01-02,570,Dota 2,648021.0,27474.0,1.00,0.85,149
2,2025-01-03,570,Dota 2,686353.0,34790.0,0.99,0.83,168
3,2025-01-04,570,Dota 2,710477.0,33816.0,0.92,0.82,148
4,2025-01-05,570,Dota 2,695257.0,33035.0,0.93,0.80,158


In [ ]:
#---------------------------------------------------------------------------------------------------------

In [4]:
def check_stationarity(series, alpha = 0.05):
    """
    Runs the Augmented Dickey-Fuller test on a pandas Series,
    dropping any NaN values prior to calculation.
    """
    clean_series = series.dropna()
    if len(clean_series) < 10:
        return {"is_stationary": False, "p_value": None}
    
    result = adfuller(clean_series, autolag='AIC')
    p_value = result[1]
    
    return {
        "is_stationary": p_value <= alpha,
        "p_value": p_value,
        "adf_statistic": result[0]
    }

In [5]:
metrics = ['player_count', 'twitch_count', 'reddit_sentiment', 'post_count', 'review_sentiment']
stationarity_results = []

# Group by game and evaluate each metric
for appid, group in panel_data.groupby('appid'):
    for metric in metrics:
        res = check_stationarity(group[metric])
        stationarity_results.append({
            'game': appid,
            'metric': metric,
            'is_stationary': res['is_stationary'],
            'p_value': res['p_value']
        })

df_adf_summary = pd.DataFrame(stationarity_results)

# Quick sanity check: How many series are non-stationary?
non_stationary_count = (df_adf_summary['is_stationary'] == False).sum()
print(f"Total non-stationary series found: {non_stationary_count} out of {len(df_adf_summary)}")

Total non-stationary series found: 103 out of 480


In [6]:
for metric in metrics:
    # 1. Standard First-Order Difference
    panel_data[f'{metric}_diff'] = panel_data.groupby('appid')[metric].diff()
    
    # 2. Alternatively, Log Difference / Percentage Change (often better for volume metrics)
    # df[f'{metric}_pct'] = df.groupby('game')[metric].pct_change()

# Drop the resulting NaN values generated by differencing (the first row of each game)
df_clean = panel_data.dropna().copy()

In [7]:
retest_results = []

for appid, group in df_clean.groupby('appid'):
    for metric in metrics:
        diff_metric = f'{metric}_diff'
        res = check_stationarity(group[diff_metric])
        retest_results.append({
            'game': appid,
            'metric': diff_metric,
            'is_stationary': res['is_stationary'],
            'p_value': res['p_value']
        })

df_retest_summary = pd.DataFrame(retest_results)

# Confirm all p-values are now below 0.05
print(df_retest_summary['is_stationary'].value_counts())

is_stationary
True     479
False      1
Name: count, dtype: int64


In [17]:
# Define all 5 stationary differenced metrics
var_metrics = [
    'player_count_diff', 
    'twitch_count_diff', 
    'post_count_diff',
    'reddit_sentiment_diff',
    'review_sentiment_diff'
]

max_lags = 7
summary_results = []

df_clean['date'] = pd.to_datetime(df_clean['date'])

for appid, group in df_clean.groupby('appid'):
    var_data = group.set_index('date')[var_metrics].dropna()
    var_data.index = pd.DatetimeIndex(var_data.index).to_period('D')
    
    if len(var_data) < (max_lags * 5):
        continue
        
    try:
        model = VAR(var_data)
        lag_selection = model.select_order(maxlags=max_lags)
        optimal_lag = lag_selection.aic
        
        results = model.fit(optimal_lag)
        is_stable = results.is_stable()
        
        row_dict = {'appid': appid, 'optimal_lag': optimal_lag, 'is_stable': is_stable}
        
        # Loop through every pair: Cause (X) -> Effect (Y)
        for caused in var_metrics:
            for causing in var_metrics:
                if caused != causing:
                    gc = results.test_causality(caused, causing, kind='f')
                    # Save p-value and boolean significance flag
                    col_name = f"{causing}_causes_{caused}"
                    row_dict[f"p_{col_name}"] = gc.pvalue
                    row_dict[f"sig_{col_name}"] = gc.pvalue < 0.05
                    
        summary_results.append(row_dict)
        
    except Exception as e:
        warnings.warn(f"Could not fit VAR for {appid}: {str(e)}")
        continue

df_var_summary = pd.DataFrame(summary_results)

In [18]:
df_var_summary['is_stable'].value_counts()

is_stable
True    96
Name: count, dtype: int64

In [19]:
print(df_var_summary['optimal_lag'].value_counts())

optimal_lag
7    82
6    11
4     2
5     1
Name: count, dtype: int64


In [21]:
# Clean labels for presentation
readable_labels = {
    'player_count_diff': 'Player Count',
    'twitch_count_diff': 'Twitch Count',
    'post_count_diff': 'Reddit Post Count',
    'reddit_sentiment_diff': 'Reddit Sentiment',
    'review_sentiment_diff': 'Steam Review Sentiment'
}

# Create an empty DataFrame for the matrix
matrix_data = {label: {} for label in readable_labels.values()}

for caused_raw, caused_label in readable_labels.items():
    for causing_raw, causing_label in readable_labels.items():
        if caused_raw == causing_raw:
            matrix_data[caused_label][causing_label] = "—"  # Self-causality is not tested
        else:
            col_name = f"sig_{causing_raw}_causes_{caused_raw}"
            percentage = (df_var_summary[col_name].mean() * 100).round(2)
            matrix_data[caused_label][causing_label] = f"{percentage}%"

# Convert to DataFrame
# Rows = Effect (Caused Variable), Columns = Cause (Causing Variable)
pairwise_matrix = pd.DataFrame(matrix_data)
pairwise_matrix.index.name = "Effect (Dependent Variable ↓) / Cause (Predictor Variable →)"

pairwise_matrix

,Player Count,Twitch Count,Reddit Post Count,Reddit Sentiment,Steam Review Sentiment
Effect (Dependent Variable ↓) / Cause (Predictor Variable →),,,,,
Player Count,—,38.54%,54.17%,4.17%,2.08%
Twitch Count,35.42%,—,18.75%,1.04%,5.21%
Reddit Post Count,60.42%,30.21%,—,4.17%,3.12%
Reddit Sentiment,3.12%,5.21%,7.29%,—,9.38%
Steam Review Sentiment,5.21%,9.38%,4.17%,10.42%,—
